# Chapter 06 · Mathematics: Indexing, permutations, and complete averages

[Chapter guide](README.md) · [Problem-to-mathematics guide](../MATHEMATICS.md) · [Course index](../README.md)

**Status: optional reference for later; not yet reviewed.** The programming lessons are already covered. Continue the programming-led mentorship without needing to complete this companion first. Use small numerical explanations when helpful, and revisit formal notation gradually.

Read one section at a time: words → a small example → symbols → code → your explanation. Symbols are shorthand for ideas you already used. You do not need to memorize all the formulas before continuing.

**Prerequisites:** Examples, features, averages, and update counts.

The runnable examples use only Python and the installed PyTorch. Start in a fresh kernel and run Setup, then proceed in order. Each example keeps its calculations small and supplies its own data. Code cells are saved unexecuted so you can make your own predictions first.

## Contents

1. [1. A dataset is a collection of matching pairs](#pairs)
2. [2. Round batch counts and retain dimensions](#batch-counts)
3. [3. Shuffle with one shared permutation](#permutation)
4. [4. Weight batch means by actual batch sizes](#weighted-loss)

## Setup

In [ ]:
import torch

<a id="pairs"></a>
## 1. A dataset is a collection of matching pairs


Write an example as the pair $(\mathbf x_i,y_i)$: input features and their matching target. A dataset contains $N$ such examples. The index $i$ chooses which pair to retrieve.

Our four input rows make a matrix $X$ of shape `[4, 2]`; targets have shape `[4, 1]`. `TensorDataset(X, y)[2]` retrieves the third input `(0, 1)` and target `3`, because Python indexes start at zero. It retrieves the pair, not two unrelated rows.

**Connection:** programming Lesson 01. **Before running:** what input–target pair should index `3` retrieve?


**My prediction or explanation:** _write here_

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# One index selects the input row and its corresponding target row.
x_dataset = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [2., 4.]])
y_dataset = torch.tensor([[2.], [5.], [3.], [12.]])
dataset = TensorDataset(x_dataset, y_dataset)
example_x, example_y = dataset[3]
print("Input:", example_x.tolist())
print("Matching target:", example_y.tolist())

<a id="batch-counts"></a>
## 2. Round batch counts and retain dimensions


If $N$ examples are not divisible by batch size $B$, we must choose whether to keep the smaller final batch.

$$\text{batches kept}=\left\lceil N/B\right\rceil\quad\text{when drop\_last=False}$$

$$\text{batches kept}=\left\lfloor N/B\right\rfloor\quad\text{when drop\_last=True}$$

The ceiling brackets $\lceil\;\rceil$ mean **round upward**. The floor brackets $\lfloor\;\rfloor$ mean **round downward**. For $4/3=1.333\ldots$, rounding upward gives two batches; rounding downward gives one.

With `drop_last=False`, the total returned example count is $N$. With `drop_last=True`, it is $B\lfloor N/B\rfloor$: full batches only. The stored dataset size does not change.

A batch with $m$ examples and two features has input shape `[m, 2]`, and our targets have shape `[m, 1]`. Even when $m=1$, the input batch is `[1, 2]`, not `[2]`. One input retrieved without batching has shape `[2]`.

With one optimizer step per batch, multiply the returned batch count by the number of epochs to count updates.

**Connection:** programming Lessons 02–04 and 06. **Before running:** predict the counts and shapes when keeping versus skipping the final batch at batch size three.


**My prediction or explanation:** _write here_

In [ ]:
import math
from torch.utils.data import TensorDataset, DataLoader

# Recreate data so this example can be read independently of the preceding one.
x_dataset = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [2., 4.]])
y_dataset = torch.tensor([[2.], [5.], [3.], [12.]])
dataset = TensorDataset(x_dataset, y_dataset)
for drop in [False, True]:
    loader = DataLoader(dataset, batch_size=3, shuffle=False, drop_last=drop)
    formula_count = math.floor(4 / 3) if drop else math.ceil(4 / 3)
    sizes = [inputs.shape[0] for inputs, targets in loader]
    print("drop_last:", drop, "| formula count:", formula_count,
          "| loader count:", len(loader), "| sizes:", sizes,
          "| examples returned:", sum(sizes))

<a id="permutation"></a>
## 3. Shuffle with one shared permutation


A **permutation** is a rearrangement of a collection's order. Let $\pi$ (pi) represent the ordered indexes chosen for a pass. For example, `[1, 0, 3, 2]` means retrieve the second example, first example, fourth example, then third example.

Apply the **same indexes** to both inputs and targets: $(\mathbf x_{\pi(i)},y_{\pi(i)})$. Shuffling input and target rows separately would break their correspondence.

With batch size three and dropping the final incomplete batch, that example order returns indexes `[1, 0, 3]` and skips index `2`. Thus the skipped example need not be the original fourth row. A random permutation can repeat; shuffling does not guarantee a different omitted example on every pass.

The code uses an explicit example permutation so the arithmetic is easy to see. The training notebook's `shuffle=True` lets `DataLoader` choose the random order instead.

**Connection:** programming Lesson 05. **Before running:** which target belongs to each rearranged input row?


**My prediction or explanation:** _write here_

In [ ]:
# Apply one shared ordering to preserve the input-target relationship.
x_dataset = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [2., 4.]])
y_dataset = torch.tensor([[2.], [5.], [3.], [12.]])
permutation = torch.tensor([1, 0, 3, 2])
ordered_inputs = x_dataset[permutation]
ordered_targets = y_dataset[permutation]
print("Rearranged inputs:", ordered_inputs.tolist())
print("Matching targets:", ordered_targets.flatten().tolist())
print("Returned targets in the full batch:", ordered_targets[:3].flatten().tolist())
print("Skipped target:", ordered_targets[3].item())

<a id="weighted-loss"></a>
## 4. Weight batch means by actual batch sizes


For one fixed model and one output per example, its whole-dataset mean loss weights each example equally. If batch $j$ contains $m_j$ examples and has mean loss $L_j$, combine batches as:

$$L_{\mathrm{all}}=\frac{\sum_j m_jL_j}{\sum_j m_j}$$

Here $j$ counts batches. Multiply each batch mean by its actual size, add those sums, then divide by the total example count.

Suppose a fixed model has squared errors `[1, 1, 1, 9]`. Batches of sizes three and one have means `1` and `9`. Averaging those means gives `5`, which gives the single final example too much weight. The correct whole-dataset mean is $(3\times1+1\times9)/4=3$.

This equality assumes errors evaluated at the same fixed parameters. A training loop that changes parameters between batches mixes different model states if it accumulates each batch's training-time loss. Our Chapter 06 training lesson instead reevaluated all examples at the final parameters after each epoch.

**Connection:** the evaluation after each epoch in programming Lesson 06. This extends that lesson's mean-loss reasoning.

**Before running:** which average treats all four examples equally?


**My prediction or explanation:** _write here_

In [ ]:
# Illustrative squared errors from one fixed model, not a new training run.
squared_errors = torch.tensor([1., 1., 1., 9.])
first_batch_mean = squared_errors[:3].mean()
last_batch_mean = squared_errors[3:].mean()
unweighted_mean_of_means = (first_batch_mean + last_batch_mean) / 2
weighted_mean = (3 * first_batch_mean + 1 * last_batch_mean) / 4
print("Unweighted mean of batch means:", unweighted_mean_of_means.item())
print("Size-weighted mean:", weighted_mean.item())
print("Direct whole-dataset mean:", squared_errors.mean().item())

**My observation:** _write here_

Explain which things change when shuffling or dropping an incomplete batch: retrieval order and returned examples. Explain which things stay fixed: stored dataset pairs and their count.